# Interactive Jupyter Visualization via `TmapViz`

This notebook shows how to configure `TmapViz` for interactive Jupyter
exploration (`show()` / `to_widget()`) and HTML export (`write_html()`).

Key features demonstrated:
- Color layout switching (continuous & categorical)
- Lasso selection → export selected indices
- `to_dataframe()` for pandas interop
- HTML export from the same `TmapViz` object

## 1. Imports


In [1]:
from importlib.metadata import PackageNotFoundError, version
try:
    print("jupyter-scatter:", version("jupyter-scatter"))
except PackageNotFoundError:
    print("jupyter-scatter is not installed")


jupyter-scatter: 0.22.2


## 2. Create fingerprints and calculate some properties 
You don'really need to worry about this part.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

# Check RDKit availability
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdFingerprintGenerator, rdMolDescriptors
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False

from tmap import LSHForest, MinHash
from tmap.layout import LayoutConfig, ScalingType, layout_from_lsh_forest
from tmap.visualization import TmapViz


def smiles_to_fingerprints(
    smiles_list: list[str],
    radius: int = 2,
    n_bits: int = 2048,
) -> tuple[np.ndarray, list[int], list]:
    """
    Convert SMILES strings to Morgan fingerprints.

    Args:
        smiles_list: List of SMILES strings
        radius: Morgan fingerprint radius (default 2 = ECFP4)
        n_bits: Number of bits in fingerprint

    Returns:
        fingerprints: numpy array of shape (n_valid, n_bits)
        valid_indices: indices of successfully parsed SMILES
        mols: list of RDKit molecule objects
    """
    if not RDKIT_AVAILABLE:
        raise ImportError("RDKit is required: pip install rdkit")

    # Use the newer MorganGenerator API
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

    fingerprints = []
    valid_indices = []
    mols = []

    for i, smi in tqdm(enumerate(smiles_list),
                       desc='Generating Fingerprints', total=len(smiles_list)):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fp = morgan_gen.GetFingerprintAsNumPy(mol)
            fingerprints.append(fp.astype(np.uint8))
            valid_indices.append(i)
            mols.append(mol)
    return np.array(fingerprints), valid_indices, mols


def compute_molecular_properties(mols: list) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute molecular properties for coloring."""
    mw = [Descriptors.MolWt(mol) for mol in mols] # type: ignore
    logp = [Descriptors.MolLogP(mol) for mol in mols] # type: ignore
    n_rings = [rdMolDescriptors.CalcNumRings(mol) for mol in mols]
    return np.array(mw), np.array(logp), np.array(n_rings)

df = pd.read_csv('../examples/cluster_65053.csv')

fingerprints, valid_indices, mols = smiles_to_fingerprints(df.smiles.to_list())
mw, logp, n_rings = compute_molecular_properties(mols)
print(f"fingerprints shape: {fingerprints.shape}")
print(f"average active bits: {fingerprints.sum(axis=1).mean():.1f}")

## 3. Encode + index + compute layout


There are two ways to do this. The verbose way (`MinHash` → `LSHForest` → `layout_from_*`) gives you full control. The `TMAP` class wraps it in a sklearn-style API.

In [3]:
# The short way 
from tmap import TMAP
tm = TMAP()
x, y, s, t= tm.fit_transform(fingerprints)
print(f"nodes: {len(x)}, edges: {len(s)}")


nodes: 64098, edges: 64097


In [4]:
# Or the long way
mh = MinHash()
signatures = mh.batch_from_binary_array(fingerprints)

lsh = LSHForest()
lsh.batch_add(signatures)
lsh.index()

x, y, s, t = layout_from_lsh_forest(lsh)
n = len(x)

print(f"signature shape: {signatures.shape}")
print(f"nodes: {n}, edges: {len(s)}")


signature shape: (64098, 128)
nodes: 64098, edges: 64097


In [ ]:
# For a quick visualization

viz = TmapViz()
viz.set_points(x,y) # Set x, y coordinates
viz.background_color = "#FFFFFF"
# Add some color layouts -you can later choose between them- 
viz.add_color_layout("Molecular Weight", mw.tolist())
viz.add_color_layout("Number of Rings", categorical=True, values= n_rings.tolist(), color='tab10')
viz.add_color_layout("LogP", logp.tolist(), categorical=False, add_as_label=True)
# Shows in the tooltip
viz.add_label("smiles", df.smiles.tolist()) 
viz.show()


# Widget
If you want to access all functionalities like selection or zoom, then create a widget

In [ ]:
# Basically the same as before, only now we can do much more cool stuff
widget =viz.to_widget()
widget.show()


In [ ]:
# We can try other functionalities like exporting the index of selected clusters!
# First, long press the left click of your mouse and then drag an area to select some points

#We now can call widget.selection() to export the index 
widget.selection()


array([46812, 48225, 18462, 45371, 51431, 45372, 52015, 41415, 51908,
       55664, 30236, 48120, 48231, 51438, 35833, 35832, 35819, 56827,
       56828, 48156, 48079, 44818, 54260, 54159, 48289, 56843, 18778,
       54189, 56812, 48233, 48160, 54281, 30196, 18777, 54188, 54187,
       48262, 48209], dtype=uint32)

## Pandas dataframe
but what good is the index if we don't have a pandas dataframe with the rest of the values we added? 

In [ ]:
df = viz.to_dataframe()
df.head()


,Molecular Weight,Number of Rings,LogP,smiles,_tmap_x,_tmap_y
0,395.837,3,3.6688,C=C1CCN(C(=O)C2=CC=C(Cl)C=N2)C(C(=O)N2CCC(=C(F...,0.552632,-0.768832
1,348.349,3,2.4556,C=C1CCN(C(=O)C2=CC=C(F)C(F)=C2)C(C(=O)N2CC=CCO...,0.254497,-0.798056
2,350.365,3,1.9844,C=C1CCN(C(=O)C2=CC=C(F)C(F)=C2)C(C(=O)N2CCOCC2)C1,0.252418,-0.792052
3,378.469,3,2.6560,C=C1CCN(C(=O)C2=CC=C(F)C(O)=C2)C(C(=O)N2CCSCC2...,0.255245,-0.792482
4,380.485,3,2.6674,C=C1CCN(C(=O)C2=CC=C(F)S2)C(C(=O)N2CCC(O)CC(C)...,0.270149,-0.793297


In [ ]:
# """Enamine REAL TMAP — scalability test with chemistry utilities."""

import os
import time

import numpy as np

from tmap import TMAP
from tmap.layout import LayoutConfig
from tmap.utils.chemistry import fingerprints_from_smiles, molecular_properties

N = 40_000 
FP_BITS = 1024
FP_RADIUS = 2
DATA_PATH = "../data/20M_smiles.txt"


print(f"=== Enamine {N // 1_000_000}M TMAP ===\n")

# ------------------------------------------------------------------
# 1. Load SMILES
# ------------------------------------------------------------------
print(f"Loading {N:,} SMILES from {DATA_PATH}...")
t0 = time.time()

smiles: list[str] = []
with open(DATA_PATH) as f:
    for i, line in enumerate(f):
        if i >= N:
            break
        smiles.append(line.strip())

print(f"  Loaded {len(smiles):,} SMILES in {time.time() - t0:.1f}s\n")

# ------------------------------------------------------------------
# 2. Compute fingerprints
# ------------------------------------------------------------------
print(f"Computing Morgan fingerprints (radius={FP_RADIUS}, {FP_BITS} bits)...")
t0 = time.time()
fps, valid_idx = fingerprints_from_smiles(
    smiles, fp_type="morgan", radius=FP_RADIUS, n_bits=FP_BITS
)
print(f"  {fps.shape[0]:,} valid / {len(smiles):,} total in {time.time() - t0:.1f}s\n")

# ------------------------------------------------------------------
# 3. Compute molecular properties
# ------------------------------------------------------------------
print("Computing molecular properties (MW, LogP, NumRings)...")
t0 = time.time()
props = molecular_properties(smiles)
print(f"  Done in {time.time() - t0:.1f}s\n")

# Filter properties to valid indices only
valid_smiles = [smiles[i] for i in valid_idx]
mw = props["mw"][valid_idx]
logp = props["logp"][valid_idx]
n_rings = props["n_rings"][valid_idx]

print(f"Fitting TMAP on {fps.shape[0]:,} fingerprints...")
t0 = time.time()

cfg = LayoutConfig()
cfg.node_size = 1 / 30
cfg.deterministic = True
cfg.seed = 42

model = TMAP(metric="jaccard", n_neighbors=20, n_permutations=512, kc=200, layout_config=cfg)
x, y, s, t_edges = model.fit_transform(fps)
print(f"  Layout done in {time.time() - t0:.1f}s\n")

# ------------------------------------------------------------------
# 5. Build visualization
# ------------------------------------------------------------------
print("Building TmapViz...")
t0 = time.time()
viz = model.to_tmapviz()
viz.title = f"Enamine_{N // 1_000_000}M"
viz.add_smiles(valid_smiles)
viz.add_color_layout("MW", mw.tolist(), categorical=False, color="viridis")
viz.add_color_layout("LogP", logp.tolist(), categorical=False, color="plasma")

# NumRings as categorical (discrete integer values)
n_rings_str = [str(int(r)) if np.isfinite(r) else "?" for r in n_rings]
viz.add_color_layout("NumRings", n_rings_str, categorical=True, color="tab20")
print(f"  Built in {time.time() - t0:.1f}s\n")

# ------------------------------------------------------------------
# 6. Export + serve
# ------------------------------------------------------------------
print("Exporting HTML (binary mode)...")
t0 = time.time()
outpath = viz.write_html(f"examples/enamine_{N // 1_000_000}m_tmap.html", mode="binary")
fsize = os.path.getsize(outpath)
print(f"  Written to {outpath} ({fsize / 1e6:.1f} MB) in {time.time() - t0:.1f}s\n")

print("Starting serve mode on port 8052...")
print("Open http://127.0.0.1:8052")
print("Press Ctrl+C to stop.\n")
viz.serve(port=8052, open_browser=True)

# Export the same `TmapViz` config to HTML
We can also export the TMAP to a html file to share or visualize in your browser

In [14]:
output_path = viz.write_html("./")
print("Saved:", output_path)


Saved: MyTMAP.html
